In [ ]:
import os
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

#### helpers

In [ ]:
def parse_datetime_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

def calculate_age(chart_date, date_of_birth):
    if len(chart_date) != len(date_of_birth):
        return 'arrays of different length'
    ages = []
    for i in range(len(chart_date)):
        years = int(chart_date[i][:4]) - int(date_of_birth[i][:4])
        months = int(chart_date[i][5:7]) - int(date_of_birth[i][5:7])
        days = int(chart_date[i][8:10]) - int(date_of_birth[i][8:10])
        if months < 0:
            years -= 1
        elif months == 0 and days < 0:
            years -= 1
        ages.append(years)
    return ages

In [ ]:
ANTIBIOTIC_KEYWORDS = [
    "penicillin", "ampicillin", "amoxicillin", "piperacillin", "tazobactam",
    "oxacillin", "nafcillin", "dicloxacillin",
    "cephalexin", "cefazolin", "cefuroxime", "cefoxitin",
    "cefotaxime", "ceftriaxone", "ceftazidime", "cefepime", "ceftaroline",
    "meropenem", "imipenem", "ertapenem", "doripenem",
    "ciprofloxacin", "levofloxacin", "moxifloxacin",
    "azithromycin", "clarithromycin", "erythromycin",
    "vancomycin", "linezolid", "tedizolid",
    "doxycycline", "tigecycline", "clindamycin", "metronidazole",
    "trimethoprim", "sulfamethoxazole", "cotrimoxazole", "co-trimoxazole", "bactrim",
    "gentamicin", "tobramycin", "amikacin",
    "colistin", "polymyxin b"]
ANTIBIOTIC_REGEX = re.compile("|".join([re.escape(k) for k in ANTIBIOTIC_KEYWORDS]), flags=re.IGNORECASE)

In [ ]:
ITEMS = {
    "temp_c":   [678, 223761],         
    "temp_f":   [679, 223762],         
    "hr":       [211, 220045],
    "sbp":      [51, 220179],
    "map":      [456, 220052, 220181],
    "rr":       [618, 220210],
    "spo2":     [646, 220277]}

def tag_type(itemid):
    i = int(itemid)
    for k, ids in ITEMS.items():
        if i in ids:
            return k
    return None

#### read data

Source data can be download on `https://physionet.org/content/mimiciii/1.4/`.

In [ ]:
# set data dir
data_dir = './source_data/mimic-iii-clinical-database-1.4/'

In [ ]:
# read notes data
notes = pd.read_csv(os.path.join(data_dir, "NOTEEVENTS.csv"), usecols=["SUBJECT_ID","HADM_ID","CHARTTIME","CATEGORY","TEXT"], low_memory=False)
notes["CHARTTIME"] = pd.to_datetime(notes["CHARTTIME"], errors="coerce")
notes.rename(columns={"SUBJECT_ID": "subject_id", "HADM_ID": "hadm_id", "CHARTTIME": "charttime", "CATEGORY": "category", "TEXT": "text"}, inplace=True)

In [ ]:
# read patient data
patients = pd.read_csv(data_dir + 'PATIENTS.csv')
patients = parse_datetime_cols(patients, ["DOB","DOD","DOD_HOSP","DOD_SSN"])
patients.rename(columns={"SUBJECT_ID":"subject_id","GENDER":"sex"}, inplace=True)

In [ ]:
# hospital admissions
admissions = pd.read_csv(os.path.join(data_dir, "ADMISSIONS.csv"), usecols=["SUBJECT_ID","HADM_ID","ADMITTIME","DISCHTIME","DEATHTIME","HOSPITAL_EXPIRE_FLAG"])
admissions = parse_datetime_cols(admissions, ["ADMITTIME","DISCHTIME","DEATHTIME"])
admissions.rename(columns={"SUBJECT_ID":"subject_id","HADM_ID":"hadm_id","ADMITTIME":"admittime","DISCHTIME":"dischtime","DEATHTIME":"deathtime", 
                           "HOSPITAL_EXPIRE_FLAG":"hospital_expire_flag"}, inplace=True)

In [ ]:
# ICU stays
icustays = pd.read_csv(os.path.join(data_dir, "ICUSTAYS.csv"), usecols=["SUBJECT_ID","HADM_ID","ICUSTAY_ID","INTIME","OUTTIME"])
icustays = parse_datetime_cols(icustays, ["INTIME","OUTTIME"])
icustays.rename(columns={"SUBJECT_ID":"subject_id","HADM_ID":"hadm_id","ICUSTAY_ID":"icustay_id","INTIME":"icu_intime","OUTTIME":"icu_outtime"}, inplace=True)

In [ ]:
# prescriptions data
rx = pd.read_csv(os.path.join(data_dir, "PRESCRIPTIONS.csv"), usecols=["SUBJECT_ID","HADM_ID","ICUSTAY_ID","STARTDATE","DRUG","DRUG_NAME_POE","ROUTE"])
rx["STARTDATE"] = pd.to_datetime(rx["STARTDATE"], errors="coerce")
rx.rename(columns={"SUBJECT_ID": "subject_id", "HADM_ID": "hadm_id", "ICUSTAY_ID": "icustay_id", "STARTDATE": "starttime","DRUG": "drug","DRUG_NAME_POE": "drug_name_poe","ROUTE": "route"}, inplace=True)

#### process features

filter

In [ ]:
# first ICU stay
icustays = icustays.sort_values(["subject_id","icu_intime","hadm_id","icustay_id"])
first_icustay_idx = icustays.groupby("subject_id")["icu_intime"].idxmin()
first_icustays = icustays.loc[first_icustay_idx].copy()

In [ ]:
# add hospital admission and patient data
df = (first_icustays
    .merge(admissions, on=["subject_id","hadm_id"], how="left")
    .merge(patients, on="subject_id", how="left"))

In [ ]:
# age
df["age"] = calculate_age(df["admittime"].astype(str), df["DOB"].astype(str))
df = df[df["age"] >= 18].copy()

In [ ]:
# time window
df["t0"] = df["icu_intime"] + pd.to_timedelta(3, unit="h")
df["t0_plus_dt"] = df["t0"] + pd.to_timedelta(6, unit="h")
df = df[df["icu_outtime"] >= df["t0_plus_dt"]].copy()

In [ ]:
# store cols
keep_cols = ["subject_id","hadm_id","icustay_id","dischtime", "t0","t0_plus_dt","deathtime","hospital_expire_flag", "age","sex"]
df = df[keep_cols].copy()

#### generate treatments and outcomes

treatment

In [ ]:
# antibiotics administration
rx["drug_text"] = (rx["drug"].fillna("") + " " + rx["drug_name_poe"].fillna("")).str.lower()
abx = rx[rx["drug_text"].str.contains(ANTIBIOTIC_REGEX, na=False)].copy()
abx = abx[["subject_id","hadm_id","icustay_id","starttime","drug","drug_name_poe"]]

In [ ]:
# first antibiotics administration per patient
abx.sort_values(["subject_id","hadm_id","icustay_id","starttime"], inplace=True)
first_abx = abx.groupby(["subject_id","hadm_id","icustay_id"], as_index=False).first()
first_abx.rename(columns={"starttime":"first_abx_time"}, inplace=True)

In [ ]:
# add
df = df.merge(first_abx, on=["subject_id","hadm_id","icustay_id"], how="left")

In [ ]:
# filter + treatment
df = df[(df["first_abx_time"].isna()) | (df["first_abx_time"] >= df["t0"])].copy()
df["T"] = np.where((df["first_abx_time"].notna()) & (df["first_abx_time"] <= df["t0_plus_dt"]),1, 0)

In [ ]:
# drop helpers
df = df.drop(columns=["drug", "drug_name_poe", "first_abx_time"])

outcomes

In [ ]:
# compute time of death
death_time = df["deathtime"].copy()
missing_death = df["hospital_expire_flag"].eq(1) & df["deathtime"].isna()
death_time.loc[missing_death] = df.loc[missing_death, "dischtime"]

In [ ]:
# true outcome
df["Y"] = np.where(death_time.notna() & (death_time >= df["t0_plus_dt"]) & (death_time <= df["dischtime"]),1, 0).astype(int)

In [ ]:
# exclude death in treatment window
df["death_in_treatment_window"] = np.where(death_time.notna() & (death_time >= df["t0"]) & (death_time < df["t0_plus_dt"]),1, 0).astype(int)
df = df[df["death_in_treatment_window"] == 0].copy()

In [ ]:
# drop helpers
df = df.drop(columns=["deathtime", "hospital_expire_flag", "death_in_treatment_window", "dischtime"])

#### text

In [ ]:
# filter
cohort_keys = df[["subject_id","hadm_id","t0"]].drop_duplicates()
notes = notes.merge(cohort_keys, on=["subject_id","hadm_id"], how="inner")
notes = notes[notes["charttime"].notna() & (notes["charttime"] <= notes["t0"])]
notes = notes[~notes["category"].str.contains("discharge", case=False, na=False)]

In [ ]:
# concat notes
notes_agg = (notes.groupby(["subject_id","hadm_id"]).agg(
    text=("text", lambda x: "\n\n".join(x.dropna().astype(str))[:200_000])).reset_index())

In [ ]:
# add text
df = df.merge(notes_agg, on=["subject_id","hadm_id"], how="left")

#### add tabular annotations

In [ ]:
# read in chuncks
chartevents_path = os.path.join(data_dir, "CHARTEVENTS.csv")
usecols = ["SUBJECT_ID","HADM_ID","ICUSTAY_ID","ITEMID","CHARTTIME","VALUENUM","VALUEUOM"]
chunksize = 1_000_000

In [ ]:
# filter on patient ids
cohort_keys = df[["subject_id","hadm_id","icustay_id","t0"]].drop_duplicates()
t0_by_icu = cohort_keys.set_index("icustay_id")["t0"].to_dict()
icu_set = set(t0_by_icu.keys())

In [ ]:
# init collector
filtered_chunks = []

# loop over chunks
for chunk in tqdm(pd.read_csv(chartevents_path, usecols=usecols, chunksize=chunksize, low_memory=False),
    desc="Reading chartevents",
    unit="chunk"):
    
    # rename and filter
    chunk.rename(columns=str.lower, inplace=True)
    tmp = chunk["icustay_id"].isin(icu_set)
    chunk = chunk.loc[tmp].copy()
    if chunk.empty:
        continue

    # add t0 and convert charttime
    chunk["t0"] = chunk["icustay_id"].map(t0_by_icu)
    chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")

    # filter
    chunk = chunk[chunk["charttime"].notna() & (chunk["charttime"] <= chunk["t0"])]
    chunk = chunk[chunk["valuenum"].notna()]
    filtered_chunks.append(chunk)
    
# concat
chart_df = pd.concat(filtered_chunks, ignore_index=True)

In [ ]:
chart_df = ce

In [ ]:
# extract vitals of interest
chart_df["vital_type"] = chart_df["itemid"].map(tag_type)
chart_df = chart_df[chart_df["vital_type"].notna()].copy()

temperature features

In [ ]:
# temperature
is_temp = chart_df["itemid"].isin(ITEMS["temp_c"] + ITEMS["temp_f"])
tmp = chart_df[is_temp].copy()
uom = tmp["valueuom"].astype(str).str.upper()

# fahrenheit masks
is_f_by_item = tmp["itemid"].isin(ITEMS["temp_f"])
is_f_by_uom  = uom.str.contains("F", na=False)
looks_f_val  = tmp["valueuom"].isna() & (tmp["valuenum"] >= 79)
is_f = is_f_by_item | is_f_by_uom | looks_f_val

# convert to celcius
tmp.loc[is_f, "valuenum"] = (tmp.loc[is_f, "valuenum"] - 32.0) * (5.0/9.0)
tmp["vital_type"] = "temp_c"
tmp = tmp[(tmp["valuenum"] >= 25.0) & (tmp["valuenum"] <= 45.0)]

# store
chart_df.loc[tmp.index, "valuenum"] = tmp["valuenum"]
chart_df.loc[tmp.index, "vital_type"] = tmp["vital_type"]

last vitals only

In [ ]:
# get last virals only
grp = ["subject_id","hadm_id","icustay_id"]
last_vals = (chart_df.sort_values(["subject_id","hadm_id","icustay_id","vital_type","charttime"])
    .groupby(grp + ["vital_type"], as_index=False)
    .tail(1)[grp + ["vital_type","valuenum"]]
    .pivot(index=grp, columns="vital_type", values="valuenum")
    .rename(columns={"temp_c":"temp_c_last","hr":"hr_last","sbp":"sbp_last","map":"map_last","rr":"rr_last","spo2":"spo2_last"}).reset_index())

In [ ]:
# select
vital_cols_keep = ["subject_id","hadm_id","icustay_id", "temp_c_last", "hr_last","rr_last","spo2_last","sbp_last","map_last"]
last_vals = last_vals[vital_cols_keep]

In [ ]:
# merge and store
df = df.merge(last_vals, on=["subject_id","hadm_id","icustay_id"], how="left")
df = df.replace({'F': 0, 'M': 1})
df = df[['subject_id', 'hadm_id', 'icustay_id', 'age', 'sex', 'temp_c_last', 'hr_last', 'rr_last', 'spo2_last', 'sbp_last', 'map_last', 'T', 'Y', 'text']]
df.to_csv('./datasets/mimic_real.csv')